<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/%E0%B8%AD%E0%B8%B1%E0%B8%99%E0%B8%AD%E0%B8%B1%E0%B8%99/Cooperative_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def generate_transaction_data(txn_id):
    """ฟังก์ชันสุ่มคิว 40 รายการต่อวัน (รับแค่ txn_id)"""
    base_date = datetime(2026, 8, 22)
    day_idx = 0
    total_items = 0

    while True:
        items_today = 40

        # เช็คว่า txn_id นี้ยังอยู่ในโควตาสะสมของวันนี้หรือไม่
        if txn_id <= total_items + items_today:
            queue_num = txn_id - total_items
            current_date = base_date + timedelta(days=day_idx)

            return {
                "วันที่": current_date.strftime("%d/%m/%Y"),
                "หมายเลขคิว": f"A-{queue_num:03d}",
                "queue_seq": queue_num - 1,
            }

        # ย้ายเข้ามาอยู่ใน while เพื่อขยับรอบวัน (แก้ Infinite Loop)
        total_items += items_today
        day_idx += 1

In [ ]:
class Transaction:

    def __init__(
        self,
        txn_id,
        account,
        transaction_type,
        amount,
        target_account=None,
    ):
        self.txn_id = txn_id

        # 1. เรียกใช้สุ่มคิวตาม txn_id
        date_info = generate_transaction_data(txn_id)
        self.queue_number = date_info["หมายเลขคิว"]
        self.txn_date = date_info["วันที่"]

        # 2. คำนวณเวลาตามลำดับคิวในวันนั้น
        base_start_time = datetime.strptime("08:30:00", "%H:%M:%S")
        queue_seq = date_info["queue_seq"]

        minutes_added = (queue_seq * random.randint(8, 11)) + random.randint(
            0, 2
        )
        seconds_added = random.randint(0, 59)

        actual_time = base_start_time + timedelta(
            minutes=minutes_added, seconds=seconds_added
        )
        self.time = actual_time.strftime("%H:%M:%S")

        # 3. จัดเก็บข้อมูลจาก Account
        self.account = account
        self.account_number = account.account_number
        self.customer_name = account.owner.customer_name
        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account

    def to_dict(self):
        # คำนวณดอกเบี้ยจาก Account
        interest = self.account.balance * self.account.interest_rate

        # แยกชื่อ - นามสกุล
        if isinstance(self.customer_name, (tuple, list)):
            fname, lname = self.customer_name[0], self.customer_name[1]
        else:
            parts = str(self.customer_name).split(" ", 1)
            fname = parts[0]
            lname = parts[1] if len(parts) > 1 else "-"

        # ดึงเลขบัญชีปลายทาง
        if self.target_account:
            target_acc_no = getattr(
                self.target_account, "account_number", str(self.target_account)
            )
        else:
            target_acc_no = "-"

        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "วันที่ทำรายการ": self.txn_date,
            "เวลาทำรายการ": self.time,
            "เลขบัญชี": self.account_number,
            "ชื่อ": fname,
            "นามสกุล": lname,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": target_acc_no,
            "ยอดหลังทำรายการ": round(self.account.balance, 2),
            "ดอกเบี้ยสิ้นปี (1.5%)": round(interest, 2),
            "ยอดรวมดอกเบี้ยสุทธิ": round(self.account.balance + interest, 2),
        }

##ส่วนที่ 2 — ฟังก์ชันช่วยงาน (Helper Function)

In [ ]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)
    return f"{first_name} {last_name}"


def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float"""
    return round(random.uniform(min_val, max_val), 2)


def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตรีงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"

##ส่วนที่ 2.1 — ทดสอบสุ่มชื่อ

In [ ]:
# เรียก generate_thai_name() 3 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ
for _ in range(3):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

##ส่วนที่ 2.2 — ทดสอบสุ่มยอดเงิน

In [ ]:
print("\n# เรียก random_amount() 3 ครั้ง")
for _ in range(3):
    print("ยอดเงินสุ่มได้:", format_currency(random_amount()))

##ส่วนที่ 2.3 — ทดสอบอัพเดทเบอร์โทรศัพท์

In [ ]:
# 1. ทดสอบสร้าง Instance ของ Member
demo_member = Member(member_id="M001", customer_name="สมชาย ใจดี", citizen_id="1234567890123", phone_number="0812345678")
print("=== [1] ทดสอบ Method ของ Class Meส่วนที่ 2.3 — ทดสอบอัพเดทเบอร์โทรศัพท์$0mber ===")
print(demo_member.get_info())

# 🔹 เพิ่มการทดสอบอัปเดตเบอร์ตรงนี้ 🔹
demo_member.update_phone("0896794152")
print("หลังอัปเดตเบอร์:", demo_member.get_info())

# 2. ทดสอบสร้าง Instance ของ Account โดยผูกกับ demo_member
demo_account = Account(account_number="100-1-00001-0", balance=1000.0, owner=demo_member)
print("\n=== [2] ทดสอบ Method ของ Class Account ===")
print(f"ยอดเงินเริ่มต้น: {demo_account.balance:,.2f} บาท")
print(f"ผลการฝากเงิน 500 บาท: {demo_account.deposit(500)}")
print(f"ยอดเงินหลังฝาก: {demo_account.balance:,.2f} บาท")
print(f"ผลการถอนเงิน 2,000 บาท: {demo_account.withdraw(2000)}")